In [1]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")

In [2]:
def print_message(message):  
    if message.message_type == "reasoning_message": 
        print("🧠 Reasoning: " + message.reasoning) 
    elif message.message_type == "assistant_message": 
        print("🤖 Agent: " + message.content) 
    elif message.message_type == "tool_call_message": 
        print("🔧 Tool Call: " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message": 
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message": 
        print("👤 User Message: " + message.content)

In [3]:
source = client.sources.create(
    name="employee_handbook",
    embedding="openai/text-embedding-3-small"
)
source

Source(name='employee_handbook', description=None, instructions=None, metadata=None, id='source-5af84e93-b597-4330-8780-540256d7fad9', embedding_config=EmbeddingConfig(embedding_endpoint_type='openai', embedding_endpoint='https://api.openai.com/v1', embedding_model='text-embedding-3-small', embedding_dim=2000, embedding_chunk_size=300, handle='openai/text-embedding-3-small', batch_size=None, azure_endpoint=None, azure_version=None, azure_deployment=None), vector_db_provider=None, created_by_id='user-00000000-0000-4000-8000-000000000000', last_updated_by_id='user-00000000-0000-4000-8000-000000000000', created_at=datetime.datetime(2026, 9, 16, 21, 39, 13, 282553, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 9, 16, 21, 39, 13, 282553, tzinfo=datetime.timezone.utc), organization_id='org-00000000-0000-4000-8000-000000000000')

In [4]:
job = client.sources.files.upload(
    source_id=source.id,
    file=open("handbook.pdf", "rb")
)

In [5]:
job.status

'created'

In [8]:
import time 

from letta_client import JobStatus

while job.status != 'completed' :
    job = client.jobs.retrieve(job.id)
    print(job.status)
    time.sleep(1)

In [9]:
job.metadata

{'type': 'embedding',
 'filename': 'handbook.pdf',
 'source_id': 'source-5af84e93-b597-4330-8780-540256d7fad9',
 'num_passages': 11,
 'num_documents': 1}

In [10]:
passages = client.sources.passages.list(
    source_id = source.id,
)
len(passages)

11

In [11]:
agent_state = client.agents.create(
    memory_blocks=[
        {
          "label": "human",
          "value": "My name is Sarah"
        },
        {
          "label": "persona",
          "value": "You are a helpful assistant"
        }
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-3-small"
)

In [13]:
agent_state = client.agents.sources.attach(
    agent_id=agent_state.id,
    source_id=source.id
)

In [14]:
client.agents.sources.list(agent_id=agent_state.id)

[Source(name='employee_handbook', description=None, instructions=None, metadata=None, id='source-5af84e93-b597-4330-8780-540256d7fad9', embedding_config=EmbeddingConfig(embedding_endpoint_type='openai', embedding_endpoint='https://api.openai.com/v1', embedding_model='text-embedding-3-small', embedding_dim=2000, embedding_chunk_size=300, handle='openai/text-embedding-3-small', batch_size=None, azure_endpoint=None, azure_version=None, azure_deployment=None), vector_db_provider=None, created_by_id='user-00000000-0000-4000-8000-000000000000', last_updated_by_id='user-00000000-0000-4000-8000-000000000000', created_at=datetime.datetime(2026, 9, 16, 21, 39, 13, 282553, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 9, 16, 21, 39, 13, 282553, tzinfo=datetime.timezone.utc), organization_id='org-00000000-0000-4000-8000-000000000000')]

In [15]:
passages=client.agents.passages.list(agent_id=agent_state.id)
len(passages)

11

In [16]:
response = client.agents.messages.create(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "Search archival for our company's vacation policies"
        }
    ]
)
for message in response.messages:
    print_message(message)

🧠 Reasoning: Searching for vacation policies in archival memory based on user's request.
🔧 Tool Call: archival_memory_search
{
  "query": "vacation policies",
  "page": 0,
  "start": 0,
  "request_heartbeat": true
}
🔧 Tool Return: ([{'timestamp': '2026-09-16 21:40:08.621936+00:00', 'content': "priorities. Assuch, vacationsarepermittedonlyunderthefollowingcondition: youmust provideanAI agent that matchesorsurpassesyourowncompetenciestofullyperformyourdutiesduringyourabsence.\nTheAI replacement must beequivalentlycompetent inall aspectsof yourrole, ensuringseamlesscontinuityof operations. Youarerequiredtosubmit theAI agent toyourimmediatesupervisorat least fourweekspriortoyourintendedleavedate. Thistimeframeallowsforrigoroustestingandevaluationof theAI'scapabilitiesandreliability.\nTheAI will undergocomprehensiveassessmentstoverifyitsproficiencyandeffectivenessinhandlingyourresponsibilities. Approval of theAI agent isat thesolediscretionof uppermanagement, andsubmissiondoesnot guaranteea

In [17]:
def query_birthday_db(name: str):
    """"
    This tool queries an external database to lookup 
    the birthday of someone given their name

    Args: 
      name(str): The name to look up 
    
    Returns:
       birthday(str): The birthday is mm-dd-yyyy format
       """
    test_data = {
        "bob": "03-05-1997",
        "sarah": "07-06-1993"
    }
    name = name.lower()
    if name not in test_data:
        return None
    else:
        return test_data[name]

In [18]:
birthday_tool = client.tools.upsert_from_function(func=query_birthday_db)

In [19]:
agent_state = client.agents.create(
    memory_blocks=[
        {
          "label": "human",
          "value": "My name is Sarah"
        },
        {
          "label": "persona",
          "value": "You are a agent with access to a birthday_db " \
            + "that you use to lookup information about users' birthdays."
        }
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-3-small",
    tool_ids=[birthday_tool.id],
    #tool_exec_environment_variables={"DB_KEY": "my_key"}
)

In [20]:
# send a message to the agent
response = client.agents.messages.create_stream(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "whens my bday????"
        }
    ]
)
for message in response:
    print_message(message)

🧠 Reasoning: User is asking about their birthday. I need to look it up in the birthday database but I don't have their name yet. I should ask for it.
🤖 Agent: I can help with that! What's your name?
